# 08 · Export for deployment (notebook stage 4)

**Research model, not a medical device.** MILK10k data and the derived model: CC BY-NC 4.0, attribution
"MILK study team", non-commercial use only.

Stages 1–3 of the notebook requirement are the release notebooks in `training/notebooks/` (01–02 EDA,
03 data processing, 04–06 modelling). Notebook 07 runs the released PyTorch ensemble on new lesions. This notebook
prepares the model for serving (decision P1-DEC-05: the full 15-member ensemble on ONNX Runtime, CPU, no TTA):

1. load and verify the 15 released checkpoints;
2. export each member to ONNX and check it against PyTorch;
3. write the **model package** (`artifacts/model_package/final-P1a-kg-onnx/`, outside git): ONNX files,
   preprocessing spec, tabular encoding, licence notice and a manifest with sha256 hashes;
4. check that the torch-free preprocessing in `milk10k_serving` equals the release transforms;
5. run the full ensemble through `milk10k_serving` (the code the API uses) and compare it with the stored reference
   predictions: it must stay within 0.002 (hard condition of P1-DEC-05).

**Inputs** (not in git): the 15 checkpoints in `training/models/final-P1a-kg/` and the six sample lesions built by
`training/sample/make_sample.py`. **Run:** `make export` from the repository root (environment: `make setup-notebooks`).

In [1]:
import os

os.environ["TQDM_DISABLE"] = "1"  # silence the release code's progress bars; tqdm reads it on import

import contextlib
import gc
import io
import json
import platform
import time
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import torch
from IPython.display import display

warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

import milk10k
from milk10k import inference as inf
from milk10k.config import checkpoint_names, load_config
from milk10k.dataset import MILKDataset, clinical_transforms, derm_transforms
from milk10k.evaluation import sha256
from milk10k.metrics import CLASSES
from milk10k.train import make_loader

import milk10k_serving
from milk10k_serving import package as pk
from milk10k_serving import preprocess as pp
from milk10k_serving.predictor import InvalidInput, Predictor

cfg = load_config()
cfg["inference"]["device"] = "cpu"
CPU = torch.device("cpu")
CKPT_DIR = Path(cfg["paths"]["checkpoint_dir"])
SAMPLE_DIR = Path(cfg["paths"]["sample_dir"])
REFERENCE = Path(cfg["paths"]["results_dir"]) / "predictions" / "final-P1a-kg.csv"
PACKAGE = ROOT / "artifacts" / "model_package" / "final-P1a-kg-onnx"
TOL = cfg["inference"]["parity_tolerance"]
INPUTS, OUTPUTS = list(pp.INPUT_NAMES), list(pk.OUTPUT_NAMES)


def rel(path) -> str:
    return Path(path).resolve().relative_to(ROOT).as_posix()


pd.DataFrame(
    {
        "version": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "onnx": onnx.__version__,
            "onnxruntime": ort.__version__,
            "milk10k": milk10k.__version__,
            "milk10k_serving": milk10k_serving.__version__,
        }
    }
)

,version
python,3.12.3
torch,2.14.0+cu130
onnx,1.23.2
onnxruntime,1.31.1
milk10k,1.0.0
milk10k_serving,0.1.0


## 1. Load and verify the released checkpoints

Each checkpoint's sha256 is compared with `training/models/final-P1a-kg/manifest.json`, then the checkpoint is
loaded with `milk10k.inference.load_member`, which also checks its identity (seed, fold, class order) and recipe
against `configs/final.yaml`. Members are loaded one at a time to keep memory low.

In [2]:
release_manifest = json.loads((CKPT_DIR / "manifest.json").read_text())
expected = {c["file"]: c["sha256"] for c in release_manifest["checkpoints"]}
MEMBERS = [
    (s, k, CKPT_DIR / f"{cfg['ensemble']['run_prefix']}-s{s}_fold{k}.pth")
    for s in cfg["ensemble"]["seeds"]
    for k in cfg["ensemble"]["folds"]
]
assert [p.name for _, _, p in MEMBERS] == checkpoint_names(cfg)
rows = []
for seed, fold, path in MEMBERS:
    digest = sha256(path)
    assert digest == expected[path.name], f"{path.name}: sha256 mismatch"
    model, args = inf.load_member(path, seed, fold, cfg, CPU)
    rows.append(
        {
            "checkpoint": path.name,
            "seed": seed,
            "fold": fold,
            "MiB": round(path.stat().st_size / 2**20, 1),
            "sha256": digest[:16] + "…",
            "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
        }
    )
    del model
    gc.collect()
print(f"loaded: {len(rows)} members, every sha256 matches the release manifest, identity and recipe checks passed")
pd.DataFrame(rows)

loaded: 15 members, every sha256 matches the release manifest, identity and recipe checks passed


,checkpoint,seed,fold,MiB,sha256,params_M
0,P1a-kg-s42_fold0.pth,42,0,90.8,fbc112c048243519…,23.75
1,P1a-kg-s42_fold1.pth,42,1,90.8,f7b8ee6931a8f2ed…,23.75
2,P1a-kg-s42_fold2.pth,42,2,90.8,b096c44e009d9421…,23.75
3,P1a-kg-s42_fold3.pth,42,3,90.8,657dba7ccdab597c…,23.75
4,P1a-kg-s42_fold4.pth,42,4,90.8,62cc7b9ebd8247ce…,23.75
5,P1a-kg-s43_fold0.pth,43,0,90.8,37e67d986018712b…,23.75
6,P1a-kg-s43_fold1.pth,43,1,90.8,0e6c51a22993206f…,23.75
7,P1a-kg-s43_fold2.pth,43,2,90.8,65573be1a8e1ac94…,23.75
8,P1a-kg-s43_fold3.pth,43,3,90.8,51c692c9cb2c5b29…,23.75
9,P1a-kg-s43_fold4.pth,43,4,90.8,65d8561517515e03…,23.75


## 2. Sample lesions through the release (PyTorch) path

The six sample lesions: three `intact`, and three with the clinical image removed (blank, NaN, missing file).
`sanitize_inputs` and the evaluation transforms are the release's own code; the resulting batch is the input for
the per-member export checks below.

In [3]:
meta = pd.read_csv(SAMPLE_DIR / "sample_metadata.csv")
clean, derm_ok, report = inf.sanitize_inputs(meta, SAMPLE_DIR)
assert derm_ok.all()
df6 = clean.reset_index(drop=True)
s = args["img_size"]
ds = MILKDataset(
    df6, str(SAMPLE_DIR), derm_transforms(s, train=False), clinical_transforms(s, train=False), clin_size=s
)
batches = list(make_loader(ds, args["batch_size"], 0, args["seed"]))
assert len(batches) == 1
batch = batches[0]
example = tuple(batch[name] for name in INPUTS)
df6[["isic_id", "scenario", "clinical_available", "age_approx", "sex", "anatomical_site_general", "fitzpatrick_type"]]

,isic_id,scenario,clinical_available,age_approx,sex,anatomical_site_general,fitzpatrick_type
0,ISIC_0145429,intact,1,60.0,male,head/neck,5.0
1,ISIC_0150253,intact,1,60.0,male,upper extremity,4.0
2,ISIC_0168990,intact,1,65.0,female,upper extremity,4.0
3,ISIC_0208181,clinical_blank,0,75.0,male,trunk,5.0
4,ISIC_0241487,clinical_nan,0,65.0,male,trunk,4.0
5,ISIC_0243690,clinical_missing_file,0,85.0,male,lower extremity,5.0


## 3. Export each member to ONNX

Each member is wrapped with the same softmax / sigmoid as `inference.predict_views` and exported with
`torch.onnx.export(dynamo=True)` (dynamic batch size). The ONNX Runtime output is compared with the PyTorch output on
the six sample lesions. The per-member PyTorch predictions (release `predict_views`) are kept to build the PyTorch
ensemble for section 6.

In [4]:
class Probs(torch.nn.Module):
    """Member forward + softmax over the 11 classes + sigmoid of the binary logit (fp32)."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, derm, clin, clin_mask, site, sex, fst, age, age_missing):
        logits_cls, logit_bin = self.net(derm, clin, clin_mask, site, sex, fst, age, age_missing)
        return torch.softmax(logits_cls.float(), dim=1), torch.sigmoid(logit_bin.float())


PACKAGE.mkdir(parents=True, exist_ok=True)
batch_dim = torch.export.Dim("batch", min=1, max=64)
feeds6 = {name: batch[name].numpy() for name in INPUTS}
torch_frames, export_rows = [], []
t_start = time.perf_counter()
for seed, fold, path in MEMBERS:
    model, a = inf.load_member(path, seed, fold, cfg, CPU)
    wrapped = Probs(model).eval()
    with torch.inference_mode():
        ref_cls, ref_bin = wrapped(*example)
    out = PACKAGE / path.with_suffix(".onnx").name
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        torch.onnx.export(
            wrapped,
            example,
            str(out),
            input_names=INPUTS,
            output_names=OUTPUTS,
            dynamo=True,
            dynamic_shapes={n: {0: batch_dim} for n in INPUTS},
            external_data=False,
            verbose=False,
        )
    export_s = time.perf_counter() - t0
    session = ort.InferenceSession(str(out), providers=["CPUExecutionProvider"])
    o_cls, o_bin = session.run(OUTPUTS, feeds6)
    opset = next(o.version for o in onnx.load(str(out), load_external_data=False).opset_import if o.domain == "")
    export_rows.append(
        {
            "member": out.name,
            "export_s": round(export_s, 1),
            "MiB": round(out.stat().st_size / 2**20, 1),
            "opset": opset,
            "max_abs_diff_cls": float(np.abs(o_cls - ref_cls.numpy()).max()),
            "max_abs_diff_bin": float(np.abs(o_bin - ref_bin.numpy()).max()),
            "top1_same": bool((o_cls.argmax(1) == ref_cls.numpy().argmax(1)).all()),
        }
    )
    torch_frames.append(inf.predict_views(model, batches, df6, CPU, False, inf.IDENTITY))
    del model, wrapped, session
    gc.collect()
torch_ensemble = inf.mean_predictions(torch_frames, df6.isic_id)
exports = pd.DataFrame(export_rows)
print(
    f"exported {len(exports)} members in {time.perf_counter() - t_start:.0f} s; "
    f"worst member diff vs PyTorch: {exports[['max_abs_diff_cls', 'max_abs_diff_bin']].to_numpy().max():.1e}"
)
assert exports.top1_same.all() and exports[["max_abs_diff_cls", "max_abs_diff_bin"]].to_numpy().max() < 1e-4
exports

exported 15 members in 88 s; worst member diff vs PyTorch: 2.3e-06


,member,export_s,MiB,opset,max_abs_diff_cls,max_abs_diff_bin,top1_same
0,P1a-kg-s42_fold0.onnx,4.4,91.0,20,7.152557e-07,8.493662e-07,True
1,P1a-kg-s42_fold1.onnx,3.8,91.0,20,1.132488e-06,1.311302e-06,True
2,P1a-kg-s42_fold2.onnx,3.5,91.0,20,6.556511e-07,1.221895e-06,True
3,P1a-kg-s42_fold3.onnx,4.1,91.0,20,8.046627e-07,1.013279e-06,True
4,P1a-kg-s42_fold4.onnx,3.9,91.0,20,5.066395e-07,7.748604e-07,True
5,P1a-kg-s43_fold0.onnx,3.8,91.0,20,5.960464e-07,6.556511e-07,True
6,P1a-kg-s43_fold1.onnx,4.1,91.0,20,7.450581e-07,1.549721e-06,True
7,P1a-kg-s43_fold2.onnx,4.1,91.0,20,7.152557e-07,1.013279e-06,True
8,P1a-kg-s43_fold3.onnx,4.0,91.0,20,5.960464e-07,5.364418e-07,True
9,P1a-kg-s43_fold4.onnx,4.3,91.0,20,6.556511e-07,4.768372e-07,True


## 4. Write the model package

Besides the ONNX members, the package carries everything the torch-free service needs: the preprocessing spec,
the fixed tabular encoding (copied from the release), a licence notice and `manifest.json` (sha256 and size of every
file, the ensemble members with their source checkpoints, input/output names, export versions). Per P1-DEC-09 there
is no prior row: the API rejects a request without a readable dermoscopic image with 422.

In [5]:
preprocessing = {
    "img_size": int(args["img_size"]),
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],
    "color": "decoded with OpenCV (IMREAD_COLOR), converted BGR -> RGB",
    "dermoscopic": "resize to img_size x img_size, bilinear (cv2.INTER_LINEAR)",
    "clinical": "longest side to img_size (Python round), bilinear, centre padding with 0 to img_size x img_size",
    "clinical_missing": "zeros after normalisation and clin_mask = false",
    "normalisation": "albumentations Normalize for uint8: per-channel float32 lookup table (x - 255*mean) / (255*std)",
    "layout": "float32, NCHW",
    "source": "milk10k.dataset derm_transforms / clinical_transforms (train=False), encode_tabular; "
    "inference.sanitize_inputs",
}
(PACKAGE / "preprocessing.json").write_text(json.dumps(preprocessing, indent=2) + "\n")
(PACKAGE / "tabular_encoding.json").write_bytes((CKPT_DIR / "tabular_encoding.json").read_bytes())
(PACKAGE / "NOTICE.md").write_text("""# final-P1a-kg-onnx: licence and attribution

- **Not a medical device.** Research model for the ISIC MILK10k benchmark; its output is not a diagnosis.
- **Licence: CC BY-NC 4.0, non-commercial use only.** The weights are derived from the ISIC Archive MILK10k data,
  CC BY-NC 4.0, attribution "MILK study team" (https://challenge.isic-archive.com/data/#milk10k).
- Pretrained initialisation: timm `resnet18.a1_in1k`, Apache-2.0.
- Content: the 15 members (3 seeds x 5 folds) of the released `final-P1a-kg` ensemble, exported to ONNX without
  retraining; see `manifest.json` for the source checkpoints and their sha256.
""")
members = [
    {"file": f"{p.stem}.onnx", "seed": s, "fold": k, "source_checkpoint": p.name, "source_sha256": expected[p.name]}
    for s, k, p in MEMBERS
]
files = {
    name: {"sha256": sha256(PACKAGE / name), "bytes": (PACKAGE / name).stat().st_size}
    for name in [m["file"] for m in members] + ["preprocessing.json", "tabular_encoding.json", "NOTICE.md"]
}
manifest = {
    "name": "final-P1a-kg-onnx",
    "version": "1.0.0",
    "model": milk10k.MODEL_NAME,
    "created": date.today().isoformat(),
    "format": "onnx",
    "disclaimer": milk10k_serving.DISCLAIMER,
    "licence": "CC BY-NC 4.0, non-commercial; derived from ISIC MILK10k (attribution: MILK study team); see NOTICE.md",
    "classes": list(CLASSES),
    "outputs": {"prob_bin": "P(malignant), binary head (target_binary)", "probs_cls": "11-class probabilities"},
    "malignant_classes": ["MEL", "BCC", "SCCKA", "MAL_OTH"],
    "inputs": [{"name": n, "dtype": str(feeds6[n].dtype), "shape": ["batch", *feeds6[n].shape[1:]]} for n in INPUTS],
    "ensemble": {
        "combine": "equal mean of member probabilities, then class renormalisation",
        "tta": False,
        "members": members,
    },
    "preprocessing": "preprocessing.json",
    "tabular_encoding": "tabular_encoding.json",
    "export": {
        "exporter": "torch.onnx.export(dynamo=True)",
        "opset": int(exports.opset.iloc[0]),
        "torch": torch.__version__,
        "onnx": onnx.__version__,
        "onnxruntime": ort.__version__,
    },
    "files": files,
}
(PACKAGE / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(
    f"package: {rel(PACKAGE)}/ — {len(files)} files + manifest.json, "
    f"{sum(f['bytes'] for f in files.values()) / 2**20:.0f} MiB"
)
pd.DataFrame(
    [{"file": n, "MiB": round(f["bytes"] / 2**20, 2), "sha256": f["sha256"][:16] + "…"} for n, f in files.items()]
)

package: artifacts/model_package/final-P1a-kg-onnx/ — 18 files + manifest.json, 1365 MiB


,file,MiB,sha256
0,P1a-kg-s42_fold0.onnx,91.0,107df8821a369fd6…
1,P1a-kg-s42_fold1.onnx,91.0,8b9518409cb070b8…
2,P1a-kg-s42_fold2.onnx,91.0,7d17f9ed91c29dcf…
3,P1a-kg-s42_fold3.onnx,91.0,45158c2e152094e6…
4,P1a-kg-s42_fold4.onnx,91.0,42aa5d608f70f1f7…
5,P1a-kg-s43_fold0.onnx,91.0,bdfedb1d2047cbbf…
6,P1a-kg-s43_fold1.onnx,91.0,14c9cf2bb53d00a3…
7,P1a-kg-s43_fold2.onnx,91.0,423111b6eabdcc33…
8,P1a-kg-s43_fold3.onnx,91.0,1fe0b9be6e30aca6…
9,P1a-kg-s43_fold4.onnx,91.0,c839f315eca47fde…


## 5. Torch-free preprocessing equals the release transforms

`milk10k_serving.preprocess` reimplements the evaluation transforms with OpenCV and NumPy only. It is compared
(a) on the six sample lesions, starting from the uploaded **bytes**, with the release batch from section 2, and
(b) on random synthetic images of awkward sizes (odd padding, upscaling) with the albumentations pipelines.

In [6]:
spec = pp.PreprocessSpec.from_package(PACKAGE)


def lesion_bytes(row):
    derm = (SAMPLE_DIR / row.derm_image_path).read_bytes()
    clin_rel = row.clinical_image_path if isinstance(row.clinical_image_path, str) else ""
    clin_path = SAMPLE_DIR / clin_rel if clin_rel.strip() else None
    return derm, (clin_path.read_bytes() if clin_path and clin_path.is_file() else None)


def metadata(row):
    return {k: getattr(row, k) for k in ("age_approx", "sex", "anatomical_site_general", "fitzpatrick_type")}


diffs = []
for i, row in enumerate(meta.itertuples()):
    derm_b, clin_b = lesion_bytes(row)
    feeds = pp.build_inputs(spec, pp.decode_rgb(derm_b), pp.decode_rgb(clin_b), **metadata(row))
    diffs.append(
        {
            "isic_id": row.isic_id,
            "scenario": row.scenario,
            **{
                n: float(np.abs(feeds[n][0].astype(np.float64) - batch[n][i].numpy().astype(np.float64)).max())
                for n in INPUTS
            },
        }
    )
lesion_diffs = pd.DataFrame(diffs)

rng = np.random.default_rng(0)
synthetic = []
for h, w in [(450, 600), (600, 450), (287, 289), (333, 1000), (120, 90), (288, 288)]:
    img = rng.integers(0, 256, (h, w, 3), dtype=np.uint8)
    d_ref = derm_transforms(s, train=False)(image=img)["image"].numpy()
    c_ref = clinical_transforms(s, train=False)(image=img)["image"].numpy()
    synthetic.append(
        {
            "size": f"{h}x{w}",
            "derm_max_abs_diff": float(np.abs(pp.dermoscopic(img, spec) - d_ref).max()),
            "clinical_max_abs_diff": float(np.abs(pp.clinical(img, spec) - c_ref).max()),
        }
    )
synthetic = pd.DataFrame(synthetic)
worst_pre = max(lesion_diffs[INPUTS].to_numpy().max(), synthetic.iloc[:, 1:].to_numpy().max())
print(f"worst preprocessing difference (sample lesions from bytes + synthetic images): {worst_pre:.1e}")
assert worst_pre < 1e-5
display(lesion_diffs)
synthetic

worst preprocessing difference (sample lesions from bytes + synthetic images): 0.0e+00


,isic_id,scenario,derm,clin,clin_mask,site,sex,fst,age,age_missing
0,ISIC_0145429,intact,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ISIC_0150253,intact,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ISIC_0168990,intact,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ISIC_0208181,clinical_blank,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ISIC_0241487,clinical_nan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,ISIC_0243690,clinical_missing_file,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,size,derm_max_abs_diff,clinical_max_abs_diff
0,450x600,0.0,0.0
1,600x450,0.0,0.0
2,287x289,0.0,0.0
3,333x1000,0.0,0.0
4,120x90,0.0,0.0
5,288x288,0.0,0.0


## 6. Full-ensemble parity through `milk10k_serving` (P1-DEC-05 hard condition)

The package is loaded exactly as the API will load it (`load_package`: sha256 of every file, one ONNX Runtime
session per member, 4 threads) and each sample lesion goes through `Predictor.predict` from its bytes. The three
intact lesions are compared with `training/results/predictions/final-P1a-kg.csv` using the release's `parity`
function (tolerance 0.002). All six are also compared with the PyTorch ensemble from section 3 and pass the release's
output checks. A request without a readable dermoscopic image must raise `InvalidInput` (HTTP 422).

In [7]:
t0 = time.perf_counter()
package = pk.load_package(PACKAGE, verify_sha256=True, threads=4)
load_s = time.perf_counter() - t0
predictor = Predictor(package)
rows, used = [], []
for row in meta.itertuples():
    derm_b, clin_b = lesion_bytes(row)
    pred = predictor.predict(derm_b, clin_b, **metadata(row))
    rows.append({"isic_id": row.isic_id, **pred.row()})
    used.append(pred.clinical_image_used)
onnx_sub = pd.DataFrame(rows)[inf.SUBMISSION_COLUMNS]
checks = inf.output_checks(onnx_sub, meta)
reference = pd.read_csv(REFERENCE)
intact = meta.scenario.eq("intact").to_numpy()
parity_intact = inf.parity(onnx_sub[intact], reference, TOL)
vs_torch = float(
    np.abs(
        onnx_sub.set_index("isic_id")[inf.PROBS] - torch_ensemble.set_index("isic_id").loc[onnx_sub.isic_id, inf.PROBS]
    )
    .to_numpy()
    .max()
)
rejected = []
for bad in (None, b"", b"\xff\xd8 truncated"):
    try:
        predictor.predict(bad, None)
    except InvalidInput as exc:
        rejected.append(exc.field)
derm_b, clin_b = lesion_bytes(next(meta[intact].itertuples()))
times = []
for _ in range(5):
    t1 = time.perf_counter()
    predictor.predict(
        derm_b, clin_b, age_approx=60, sex="male", anatomical_site_general="head/neck", fitzpatrick_type=4
    )
    times.append(time.perf_counter() - t1)
summary = {
    "intact lesions vs reference, max |diff|": f"{parity_intact['max_abs_diff']:.2e}",
    "within 0.002 (hard condition)": parity_intact["within_tolerance"],
    "all 6 lesions vs PyTorch ensemble, max |diff|": f"{vs_torch:.2e}",
    "output checks (6 lesions)": all(checks.values()),
    "clinical image used (6 lesions)": used,
    "unreadable / missing dermoscopic image rejected": f"{len(rejected)}/3",
    "package load incl. sha256 (s)": round(load_s, 1),
    "1 lesion, 15 members, 4 threads, median of 5 (s)": round(float(np.median(times)), 3),
}
assert parity_intact["within_tolerance"], f"P1-DEC-05 hard condition failed: {parity_intact}"
assert vs_torch < 1e-4 and len(rejected) == 3 and used == [True, True, True, False, False, False]
pd.Series(summary, name="result").to_frame()

,result
"intact lesions vs reference, max |diff|",2.55e-04
within 0.002 (hard condition),True
"all 6 lesions vs PyTorch ensemble, max |diff|",2.26e-07
output checks (6 lesions),True
clinical image used (6 lesions),"[True, True, True, False, False, False]"
unreadable / missing dermoscopic image rejected,3/3
package load incl. sha256 (s),2.3
"1 lesion, 15 members, 4 threads, median of 5 (s)",0.695


In [8]:
manifest["verification"] = {
    "date": date.today().isoformat(),
    "member_max_abs_diff_vs_pytorch": float(exports[["max_abs_diff_cls", "max_abs_diff_bin"]].to_numpy().max()),
    "preprocessing_max_abs_diff_vs_release": float(worst_pre),
    "reference": "training/results/predictions/final-P1a-kg.csv",
    "intact_sample_lesions_max_abs_diff_vs_reference": parity_intact["max_abs_diff"],
    "sample_lesions_max_abs_diff_vs_pytorch_ensemble": vs_torch,
    "tolerance": TOL,
    "passed": bool(parity_intact["within_tolerance"]),
}
(PACKAGE / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(f"verification written to {rel(PACKAGE / 'manifest.json')}: passed = {manifest['verification']['passed']}")

verification written to artifacts/model_package/final-P1a-kg-onnx/manifest.json: passed = True


## Next steps

- Publish the package folder as a GitHub Release asset (P1-SET-10); `scripts/fetch_model.sh` will download and
  verify it (P1-BSH-02).
- The FastAPI service (milestone M5) loads it with `milk10k_serving.package.load_package` and serves
  `Predictor.predict`; the parity test `tests/test_predictor_parity.py` repeats section 6 in CI.